# A9-CWO · two-channel Discord daily collector

**Scope:** exactly guild \`682670494658330644\`, channels \`1463826087841890417\` and \`1492168056858738809\`, and currently active threads whose \`parent_id\` equals one of these two channels. The bot reads no other guild messages.

Run this notebook **before** the A9-CWO daily reconciliation. Its outputs are **raw UTF-8 JSONL, unchanged image/file bytes, machine-readable candidate observations, state cursors, and a JSON handoff receipt** in the established A9 Discord Drive domain. It does **not** post to Discord, Slack or GitHub, and does not call work DONE merely because someone sent a message.

**Setup once:** add the existing authorized Discord bot to both channels with View Channel and Read Message History, enable Message Content in the Discord Developer Portal if required, create a Google Colab secret named \`DISCORD_BOT_TOKEN\`, and grant that Colab notebook access to the secret. Don't use a Discord user token/self-bot. Use the same Google account that owns the A9 Drive tree.

**First run:** a bounded 7-day bootstrap, not full-server history. Later runs resume each channel/thread cursor with a two-day lookback so edits can be discovered. The scanned window and coverage debts appear in each receipt. Only active child threads are included in v1: archived threads and image embeds without actual attachment objects are **not** claimed complete.

**Existing Drive authority:** \`12_DISCORD_SYSTEM_AND_ARCHIVES\`, stable ID \`1nTFR9xQXNwgkWRrAq1jJw7ph6rKg3C9X\`. The collector creates/reuses just one child folder \`A9_CWO_DAILY_TWO_CHANNELS\`, rather than duplicating the whole-server archive. Preserve the existing Discord full-fidelity archive and its immutable history.

**Run order:** Run Setup → Run Collector → inspect printed receipt status + Drive output → then run or allow A9-CWO to reconcile. If any source returns PARTIAL, don't claim complete capture or advance its watermark.

In [ ]:
# SETUP — Google Drive authorization and the two-source configuration
%pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib requests

from google.colab import auth, userdata
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from getpass import getpass

SCOPES = ["https://www.googleapis.com/auth/drive"]
credentials, _ = google.auth.default(scopes=SCOPES)
drive = build("drive", "v3", credentials=credentials, cache_discovery=False)

GUILD_ID = "682670494658330644"
ROOT_DRIVE_ID = "1nTFR9xQXNwgkWRrAq1jJw7ph6rKg3C9X"
PARENT_CHANNEL_IDS = ["1463826087841890417", "1492168056858738809"]
RUN_FOLDER_NAME = "A9_CWO_DAILY_TWO_CHANNELS"
BOOTSTRAP_DAYS = 7
REVISIT_DAYS = 2
MAX_PAGES_PER_SOURCE = 100
DOWNLOAD_ATTACHMENTS = True
MAX_ATTACHMENT_BYTES = 100 * 1024 * 1024
INCLUDE_ACTIVE_CHILD_THREADS = True

try:
    BOT_TOKEN = userdata.get("DISCORD_BOT_TOKEN")
except Exception:
    BOT_TOKEN = None
if not BOT_TOKEN:
    BOT_TOKEN = getpass("Paste Discord BOT token (input hidden): ").strip()
if not BOT_TOKEN:
    raise RuntimeError("Discord bot token is required. Stop; do not substitute a user token.")
assert len(set(PARENT_CHANNEL_IDS)) == 2
print("Drive connected; Discord scope: 1 guild, 2 parents. Token hidden.")

In [ ]:
# COLLECTOR — read-only Discord API; write raw/source receipts to your existing A9 Drive tree
import io, os, re, time, json, hashlib, tempfile
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import requests
from googleapiclient.http import MediaIoBaseUpload, MediaFileUpload, MediaIoBaseDownload

TZ = ZoneInfo("Asia/Kathmandu")
API = "https://discord.com/api/v10"
http = requests.Session()
http.headers.update({"Authorization": "Bot " + BOT_TOKEN, "User-Agent": "A9-CWO-DailyTwoChannelCollector/1.0"})
RUN_AT = datetime.now(timezone.utc)
RUN_ID = RUN_AT.strftime("%Y%m%dT%H%M%SZ")
DISCORD_EPOCH_MS = 1420070400000

def jsonbytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8")

def get_discord(path, params=None):
    for retry in range(6):
        res = http.get(API + path, params=params, timeout=75)
        if res.status_code == 429:
            wait = float(res.json().get("retry_after", 1))
            time.sleep(min(max(wait, 0.2), 30))
            continue
        if 500 <= res.status_code < 600 and retry < 5:
            time.sleep(min(2**retry, 16))
            continue
        if not res.ok:
            raise RuntimeError("Discord API returned HTTP %s for %s; %s" % (
                res.status_code, path, res.text[:250]))
        return res.json()
    raise RuntimeError("Discord API retries exhausted for " + path)

def after_snowflake(dt):
    ms = int(dt.timestamp() * 1000)
    return str(max(0, ms - DISCORD_EPOCH_MS) << 22)

def quote_drive(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def children(parent, name):
    query = "trashed = false and '%s' in parents and name = '%s'" % (quote_drive(parent), quote_drive(name))
    rows = []
    token = None
    while True:
        result = drive.files().list(
            q=query, fields="nextPageToken,files(id,name,mimeType,size,md5Checksum)",
            page_size=100, page_token=token, supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        rows += result.get("files", [])
        token = result.get("nextPageToken")
        if not token:
            break
    if len(rows) > 1:
        raise RuntimeError("Ambiguous duplicate Drive name in parent %s: %s" % (parent, name))
    return rows[0] if rows else None

def directory(parent, name):
    old = children(parent, name)
    if old:
        if old["mimeType"] != "application/vnd.google-apps.folder":
            raise RuntimeError("Expected folder, found existing file: " + name)
        return old["id"]
    return drive.files().create(
        body={"name": name, "mimeType": "application/vnd.google-apps.folder",
              "parents": [parent]},
        fields="id", supportsAllDrives=True).execute()["id"]

def byte_readback(file_id):
    stream = io.BytesIO()
    request = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    dl = MediaIoBaseDownload(stream, request)
    done = False
    while not done:
        _, done = dl.next_chunk()
    return stream.getvalue()

def save_bytes(parent, name, contents, mime="application/json"):
    expected = hashlib.sha256(contents).hexdigest()
    old = children(parent, name)
    if old:
        actual = byte_readback(old["id"])
        if hashlib.sha256(actual).hexdigest() != expected:
            raise RuntimeError("Existing Drive artifact conflicts; no overwrite: " + name)
        return {"id": old["id"], "sha256": expected, "bytes": len(contents), "reused": True}
    media = MediaIoBaseUpload(io.BytesIO(contents), mimetype=mime, resumable=True)
    obj = drive.files().create(
        body={"name": name, "parents": [parent]}, media_body=media,
        fields="id,size,md5Checksum", supportsAllDrives=True).execute()
    actual = byte_readback(obj["id"])
    if hashlib.sha256(actual).hexdigest() != expected:
        raise RuntimeError("Drive byte readback mismatch for " + name)
    return {"id": obj["id"], "sha256": expected, "bytes": len(contents), "reused": False}

def state_read(root):
    obj = children(root, "state_current.json")
    if not obj:
        return {"schema": "a9-cwo-discord-state-v1", "guild_id": GUILD_ID,
                "source_states": {}}, None
    parsed = json.loads(byte_readback(obj["id"]).decode("utf-8"))
    if parsed.get("guild_id") != GUILD_ID or parsed.get("schema") != "a9-cwo-discord-state-v1":
        raise RuntimeError("Existing state conflicts with requested guild/schema")
    return parsed, obj

def state_write(root, state, old):
    payload = jsonbytes(state)
    media = MediaIoBaseUpload(io.BytesIO(payload), mimetype="application/json", resumable=True)
    if old:
        obj = drive.files().update(fileId=old["id"], media_body=media,
                                   fields="id", supportsAllDrives=True).execute()
    else:
        obj = drive.files().create(body={"name": "state_current.json", "parents": [root]},
                                   media_body=media, fields="id", supportsAllDrives=True).execute()
    if hashlib.sha256(byte_readback(obj["id"])).digest() != hashlib.sha256(payload).digest():
        raise RuntimeError("State readback failed")
    return {"id": obj["id"]}

def media_archive(folder, msg, attachment):
    aid, mid = str(attachment["id"]), str(msg["id"])
    size = int(attachment.get("size") or 0)
    if size > MAX_ATTACHMENT_BYTES:
        raise RuntimeError("Attachment exceeds MAX_ATTACHMENT_BYTES: id " + aid)
    safe_name = re.sub(r"[^-._a-zA-Z0-9]+", "_", attachment.get("filename", "file"))[:96]
    name = mid + "__" + aid + "__" + safe_name
    old = children(folder, name)
    if old:
        if size and int(old.get("size") or -1) != size:
            raise RuntimeError("Existing attachment size conflict: id " + aid)
        return {"attachment_id": aid, "message_id": mid, "drive_id": old["id"],
                "provider_size": size, "reused": True, "status": "PREVIOUSLY_ARCHIVED"}
    url = attachment.get("url")
    if not url or urlparse(url).scheme != "https":
        raise RuntimeError("Attachment URL missing or non-HTTPS: " + aid)
    # CDN signatures may expire. The next run re-fetches message metadata.
    # A NEW session-less request prevents leaking the Discord Bot Authorization header to a CDN.
    host = (urlparse(url).hostname or "").lower()
    if not (host == "cdn.discordapp.com" or host == "media.discordapp.net" or host.endswith(".discordapp.net")):
        raise RuntimeError("Unexpected attachment CDN hostname for id " + aid)
    with requests.get(url, stream=True, timeout=120, headers={"User-Agent": "A9-CWO/1.0"}) as res:
        if not res.ok:
            raise RuntimeError("Attachment GET failed: HTTP %s, id %s" % (res.status_code, aid))
        with tempfile.NamedTemporaryFile(suffix=".bin") as tmp:
            sha = hashlib.sha256()
            md5 = hashlib.md5()
            count = 0
            for chunk in res.iter_content(chunk_size=1024*1024):
                if not chunk:
                    continue
                count += len(chunk)
                if count > MAX_ATTACHMENT_BYTES:
                    raise RuntimeError("Attachment size cap exceeded: " + aid)
                sha.update(chunk)
                md5.update(chunk)
                tmp.write(chunk)
            tmp.flush()
            if size and count != size:
                raise RuntimeError("Attachment size mismatch, id " + aid)
            media = MediaFileUpload(tmp.name, mimetype=attachment.get("content_type") or "application/octet-stream",
                                    resumable=True)
            created = drive.files().create(body={"name": name, "parents": [folder]},
                                           media_body=media, fields="id,size,md5Checksum",
                                           supportsAllDrives=True).execute()
    if int(created.get("size") or -1) != count or (created.get("md5Checksum") and created["md5Checksum"] != md5.hexdigest()):
        raise RuntimeError("Provider size/MD5 readback mismatch: attachment " + aid)
    return {"attachment_id": aid, "message_id": mid, "drive_id": created["id"],
            "provider_size": size, "stored_bytes": count, "sha256": sha.hexdigest(),
            "provider_md5": created.get("md5Checksum"), "reused": False, "status": "ARCHIVED"}

def fetch_pages(channel_id, lower_bound):
    collected, before = [], None
    for page_number in range(MAX_PAGES_PER_SOURCE):
        params = {"limit": 100}
        if before:
            params["before"] = before
        page = get_discord("/channels/%s/messages" % channel_id, params)
        if not page:
            return collected
        collected.extend(m for m in page if int(m["id"]) > int(lower_bound))
        earliest = min(int(m["id"]) for m in page)
        if earliest <= int(lower_bound) or len(page) < 100:
            return collected
        before = str(earliest)
    raise RuntimeError("MAX_PAGES_PER_SOURCE exceeded; cursor will not advance: " + channel_id)

def in_scope_sources():
    scoped = {}
    for parent_id in PARENT_CHANNEL_IDS:
        meta = get_discord("/channels/" + parent_id)
        if str(meta.get("guild_id")) != GUILD_ID:
            raise RuntimeError("Channel not in configured guild: " + parent_id)
        scoped[parent_id] = {"parent_id": parent_id, "channel_id": parent_id, "name": meta.get("name"), "type": meta.get("type")}
    if INCLUDE_ACTIVE_CHILD_THREADS:
        # This endpoint reads THREAD METADATA, not messages in unrelated channels.
        active = get_discord("/guilds/%s/threads/active" % GUILD_ID)
        for thread in active.get("threads", []):
            parent_id = str(thread.get("parent_id"))
            thread_id = str(thread.get("id"))
            if parent_id in PARENT_CHANNEL_IDS:
                scoped[thread_id] = {
                    "parent_id": parent_id, "channel_id": thread_id,
                    "name": thread.get("name"), "type": thread.get("type")}
    return scoped

root_meta = drive.files().get(fileId=ROOT_DRIVE_ID, fields="id,name,mimeType", supportsAllDrives=True).execute()
if root_meta["mimeType"] != "application/vnd.google-apps.folder":
    raise RuntimeError("Discord A9 root Drive ID did not resolve to a folder")
root = directory(ROOT_DRIVE_ID, RUN_FOLDER_NAME)
rawdir = directory(root, "RAW_JSONL")
mediadir = directory(root, "ATTACHMENT_BYTES")
canddir = directory(root, "CANDIDATES_JSONL")
receiptdir = directory(root, "RUN_RECEIPTS")
state, oldstate = state_read(root)
scoped = in_scope_sources()

receipt = {
 "schema": "a9-cwo-discord-receipt-v1", "run_id": RUN_ID, "guild_id": GUILD_ID,
 "target_parent_channels": PARENT_CHANNEL_IDS, "root_drive_id": root,
 "collection_started_at": RUN_AT.isoformat(), "sources": [],
 "scope_rule": "Only selected parent channels and currently active children of those parents",
 "archived_threads_covered": False, "embed_only_images_covered": False,
 "replay_lookback_days": REVISIT_DAYS, "bootstrap_days": BOOTSTRAP_DAYS
}
for source_id, meta in sorted(scoped.items()):
    record = {"channel_id": source_id, "parent_channel_id": meta["parent_id"],
              "channel_name": meta["name"], "status": "PENDING"}
    current = dict(state["source_states"].get(source_id, {}))
    last_good = current.get("highwater_message_id")
    horizon = datetime.now(timezone.utc) - timedelta(days=BOOTSTRAP_DAYS)
    if last_good:
        previous_ms = (int(last_good) >> 22) + DISCORD_EPOCH_MS
        # Never truncate a multi-week catch-up to the bootstrap 7-day window.
        horizon = datetime.fromtimestamp(previous_ms/1000, timezone.utc) - timedelta(days=REVISIT_DAYS)
    lower = after_snowflake(horizon)
    record["window_start_utc"] = horizon.isoformat()
    record["previous_highwater_message_id"] = last_good
    try:
        messages = fetch_pages(source_id, lower)
        messages.sort(key=lambda m: int(m["id"]))
        versions = dict(current.get("recent_versions", {}))
        changed, attachments, candidate_rows = [], [], []
        reused = 0
        folder = directory(mediadir, source_id)
        for msg in messages:
            key = str(msg["id"])
            # Do not hash expiring signed CDN URLs: stable content, edits and attachment IDs determine a version.
            content_signature = {
                "id": key, "content": msg.get("content"), "edited_timestamp": msg.get("edited_timestamp"),
                "author_id": msg.get("author", {}).get("id"),
                "attachments": [{"id": a.get("id"), "size": a.get("size"),
                                 "filename": a.get("filename")} for a in msg.get("attachments", [])],
                "embeds": msg.get("embeds"), "flags": msg.get("flags"),
                "reactions": msg.get("reactions")
            }
            raw_digest = hashlib.sha256(jsonbytes(content_signature)).hexdigest()
            if versions.get(key) == raw_digest:
                reused += 1
                continue
            changed.append(msg)
            for attachment in msg.get("attachments", []):
                try:
                    attachments.append(media_archive(folder, msg, attachment) if DOWNLOAD_ATTACHMENTS
                                       else {"attachment_id": str(attachment["id"]), "status": "NOT_DOWNLOADED"})
                except Exception as err:
                    attachments.append({"attachment_id": str(attachment.get("id")),
                                        "message_id": key, "status": "FAILED",
                                        "reason": str(err)[:300]})
            candidate_rows.append({
                "provider": "DISCORD", "guild_id": GUILD_ID,
                "parent_channel_id": meta["parent_id"], "channel_id": source_id,
                "message_id": key, "message_url": "https://discord.com/channels/%s/%s/%s" % (GUILD_ID, source_id, key),
                "author_id": msg.get("author", {}).get("id"), "created_at": msg.get("timestamp"),
                "edited_at": msg.get("edited_timestamp"), "text_original": msg.get("content", ""),
                "attachment_ids": [str(a.get("id")) for a in msg.get("attachments", [])],
                "project_id": None, "fact_admission": "NEEDS_RECONCILIATION"
            })
            versions[key] = raw_digest
        errors = [a for a in attachments if a.get("status") in ("FAILED", "NOT_DOWNLOADED")]
        record["scanned_messages"] = len(messages)
        record["new_or_changed_messages"] = len(changed)
        record["previously_seen_unchanged"] = reused
        record["attachments"] = len(attachments)
        record["failed_or_unarchived_attachments"] = len(errors)
        record["attachments_evidence"] = attachments
        if changed:
            suffix = source_id + "__" + RUN_ID
            raw = b"".join(jsonbytes(x) for x in changed)
            candidates = b"".join(jsonbytes(x) for x in candidate_rows)
            record["raw_jsonl"] = save_bytes(rawdir, suffix + ".jsonl", raw, "application/x-ndjson")
            record["candidate_jsonl"] = save_bytes(canddir, suffix + ".jsonl", candidates, "application/x-ndjson")
        if errors:
            record["status"] = "PARTIAL_ATTACHMENTS"
        else:
            record["status"] = "PASS"
            if messages:
                newest = str(max(int(x["id"]) for x in messages))
                current["highwater_message_id"] = str(max(int(last_good or 0), int(newest)))
            current["recent_versions"] = versions
            current["last_verified_at"] = datetime.now(timezone.utc).isoformat()
            state["source_states"][source_id] = current
            # Persist verified checkpoint after each successful source, so later source failure cannot lose progress.
            oldstate = state_write(root, state, oldstate)
    except Exception as err:
        record["status"] = "FAILED_SOURCE"
        record["error"] = str(err)[:600]
        # No cursor advance on this source.
    receipt["sources"].append(record)
receipt["collection_finished_at"] = datetime.now(timezone.utc).isoformat()
receipt["status"] = "PASS" if receipt["sources"] and all(x["status"] == "PASS" for x in receipt["sources"]) else "PARTIAL"
receipt["source_count"] = len(receipt["sources"])
receipt["messages_new_or_changed"] = sum(x.get("new_or_changed_messages", 0) for x in receipt["sources"])
receipt["attachment_failures"] = sum(x.get("failed_or_unarchived_attachments", 0) for x in receipt["sources"])
# Frozen run receipt is written only after channel-specific archival attempts.
receipt_link = save_bytes(receiptdir, "A9_CWO_DISCORD_RECEIPT__" + RUN_ID + ".json", jsonbytes(receipt))
handoff = {
 "schema": "a9-cwo-discord-handoff-v1", "provider": "DISCORD",
 "run_id": RUN_ID, "status": receipt["status"],
 "guild_id": GUILD_ID, "scope": PARENT_CHANNEL_IDS,
 "receipt_file_id": receipt_link["id"], "archive_root_drive_id": root,
 "new_or_changed_message_count": receipt["messages_new_or_changed"],
 "collection_finished_at": receipt["collection_finished_at"],
 "per_channel": [{k: item.get(k) for k in (
     "channel_id", "parent_channel_id", "status", "previous_highwater_message_id",
     "scanned_messages", "new_or_changed_messages", "failed_or_unarchived_attachments")}
   for item in receipt["sources"]]
}
oldhandoff = children(root, "handoff_current.json")
media = MediaIoBaseUpload(io.BytesIO(jsonbytes(handoff)), mimetype="application/json", resumable=True)
if oldhandoff:
    p = drive.files().update(fileId=oldhandoff["id"], media_body=media, fields="id", supportsAllDrives=True).execute()
else:
    p = drive.files().create(body={"name": "handoff_current.json", "parents": [root]},
                             media_body=media, fields="id", supportsAllDrives=True).execute()
if byte_readback(p["id"]) != jsonbytes(handoff):
    raise RuntimeError("Handoff readback failed; A9-CWO is NOT ready")
print(json.dumps({"status": receipt["status"], "run_id": RUN_ID,
    "scoped_sources": receipt["source_count"], "new_or_changed_messages": receipt["messages_new_or_changed"],
    "attachment_failures": receipt["attachment_failures"], "handoff_drive_id": p["id"],
    "receipt_drive_id": receipt_link["id"], "archive_folder": root}, indent=2))
print("Run A9-CWO only after checking this receipt. 'PARTIAL' needs reconciliation.")